In [ ]:
"""Scrap pymatgen dependent repositories from GitHub:
https://github.com/materialsproject/pymatgen/network/dependents?dependent_type=REPOSITORY


NOTE:
1. For some reason the number of stars cannot be scrapped correctly,
have to scrap the repos first, then get stars later.

2. The website showed around 2500 repos but the script only got 1500,
not sure what's the reason.
"""

In [ ]:
!uv pip install requests pandas bs4

import os
import time
from urllib.parse import urlsplit, parse_qs, urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://github.com/materialsproject/pymatgen/network/dependents"
PARAMS_BASE = {"dependent_type": "REPOSITORY"}

OUTFILE = "_pymatgen_dependents.csv"
CURSOR_FILE = "_pymatgen_dependents_cursor.txt"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (pymatgen-dependents-scraper)",
    "Accept": "text/html,application/xhtml+xml",
}


def extract_next_cursor(soup: BeautifulSoup) -> str | None:
    """Find the 'Next' pagination link and extract its dependents_after cursor."""
    for a in soup.find_all("a", href=True):
        if a.get_text(strip=True) == "Next" and "dependents_after=" in a["href"]:
            qs = parse_qs(urlsplit(a["href"]).query)
            vals = qs.get("dependents_after")
            if vals:
                return vals[0]
    return None


def extract_repo(row: BeautifulSoup) -> tuple[str | None, str | None]:
    """Extract 'owner/repo' and absolute repo URL."""
    link = row.select_one('a[data-hovercard-type="repository"]')
    if not link:
        return None, None
    full_name = (link.get_text() or "").strip()
    href = link.get("href") or ""
    repo_url = urljoin("https://github.com", href)
    return full_name, repo_url


def save_checkpoint(rows: list[dict]) -> None:
    df = pd.DataFrame(rows).drop_duplicates("full_name")
    df.to_csv(OUTFILE, index=False)


def load_existing() -> tuple[list[dict], set[str], str | None]:
    repos, seen, cursor = [], set(), None
    if os.path.exists(OUTFILE):
        df = pd.read_csv(OUTFILE)
        repos = df.to_dict("records")
        seen = set(df["full_name"].astype(str))
        print(f"Resuming: {len(df)} repos loaded from {OUTFILE}")
    if os.path.exists(CURSOR_FILE):
        cursor = (open(CURSOR_FILE, "r", encoding="utf-8").read().strip()) or None
        if cursor:
            print(f"Resuming from cursor: {cursor[:12]}…")
    return repos, seen, cursor


def main():
    all_repos, seen, cursor = load_existing()
    page_ix = 1

    while True:
        params = PARAMS_BASE.copy()
        if cursor:
            params["dependents_after"] = cursor

        print(f"\n[page {page_ix}] GET {BASE_URL} params={params}")
        try:
            r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=30)
        except requests.RequestException as e:
            print(f"Request error: {e}. Sleeping 20s…")
            time.sleep(20)
            continue

        if r.status_code in (403, 429):
            wait = int(r.headers.get("Retry-After", "20"))
            print(f"Rate-limited ({r.status_code}). Sleeping {wait}s…")
            time.sleep(wait)
            continue
        if r.status_code != 200:
            print(f"Stopped: HTTP {r.status_code}")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        rows = soup.select("div.Box-row")
        if not rows:
            print("No rows found on this page; done.")
            break

        page_new = []
        for row in rows:
            full_name, repo_url = extract_repo(row)
            if not full_name or full_name in seen:
                continue
            page_new.append({"full_name": full_name, "repo_url": repo_url})
            seen.add(full_name)

        if page_new:
            all_repos.extend(page_new)
            save_checkpoint(all_repos)
            # Show preview
            preview = page_new[:5]
            print(f"Saved {len(all_repos)} total repos → {OUTFILE}")
            print("Preview from this page:")
            for rec in preview:
                print(f"  - {rec['full_name']} {rec['repo_url']}")
        else:
            print("No NEW repos on this page.")

        next_cursor = extract_next_cursor(soup)
        if not next_cursor:
            print("No 'Next' cursor found — reached the end.")
            break

        cursor = next_cursor
        with open(CURSOR_FILE, "w", encoding="utf-8") as f:
            f.write(cursor)
        print(f"Next cursor: {cursor[:12]}… (saved to {CURSOR_FILE})")

        page_ix += 1
        time.sleep(1.1)  # polite delay

    print(f"\nDone. Final CSV: {OUTFILE}")


if __name__ == "__main__":
    main()

Resuming: 1446 repos loaded from _pymatgen_dependents.csv

[page 1] GET https://github.com/materialsproject/pymatgen/network/dependents params={'dependent_type': 'REPOSITORY'}
No NEW repos on this page.
Next cursor: NDgwNjkxMTU4… (saved to _pymatgen_dependents_cursor.txt)

[page 2] GET https://github.com/materialsproject/pymatgen/network/dependents params={'dependent_type': 'REPOSITORY', 'dependents_after': 'NDgwNjkxMTU4ODY'}
No NEW repos on this page.
Next cursor: NDc0MzgwOTUx… (saved to _pymatgen_dependents_cursor.txt)

[page 3] GET https://github.com/materialsproject/pymatgen/network/dependents params={'dependent_type': 'REPOSITORY', 'dependents_after': 'NDc0MzgwOTUxNzk'}
No NEW repos on this page.
Next cursor: NDcxMDczMDY0… (saved to _pymatgen_dependents_cursor.txt)


KeyboardInterrupt: 

In [ ]:
"""Extract number of stars/forks for each repo."""

INPUT_FILE = "_pymatgen_dependents.csv"
OUTPUT_FILE = "pymatgen_dependents_with_stars.csv"

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


def fetch_repo_info(full_name: str) -> dict:
    """Fetch stargazers_count and forks_count from GitHub API for owner/repo slug."""
    url = f"https://api.github.com/repos/{full_name}"
    headers = {
        "Accept": "application/vnd.github+json",
        "User-Agent": "pymatgen-dependents-scraper",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

    r = requests.get(url, headers=headers)

    # Handle rate limits
    if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
        reset_ts = int(r.headers.get("X-RateLimit-Reset", time.time() + 60))
        wait = max(0, reset_ts - int(time.time())) + 5
        print(f"Rate limit reached, sleeping {wait}s…")
        time.sleep(wait)
        return fetch_repo_info(full_name)

    if r.status_code != 200:
        print(f"Failed {full_name}: {r.status_code}")
        return {"stars": None, "forks": None}

    data = r.json()
    return {
        "stars": data.get("stargazers_count", 0),
        "forks": data.get("forks_count", 0),
    }


def main():
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} repos from {INPUT_FILE}")

    # Always ensure owner/repo slug is available
    df["owner_repo"] = df["repo_url"].str.replace(
        "https://github.com/", "", regex=False
    )

    # Resume if output exists
    if os.path.exists(OUTPUT_FILE):
        df_out = pd.read_csv(OUTPUT_FILE)
        if "owner_repo" not in df_out.columns:  # fix older runs
            df_out["owner_repo"] = df_out["repo_url"].str.replace(
                "https://github.com/", "", regex=False
            )
        done = set(df_out["owner_repo"])
        print(f"Resuming: already enriched {len(done)} repos")
    else:
        df_out = df.copy()
        df_out["stars"] = None
        df_out["forks"] = None
        done = set()

    for i, row in df.iterrows():
        full_name = row["owner_repo"]
        if full_name in done:
            continue

        info = fetch_repo_info(full_name)
        df_out.loc[df_out["owner_repo"] == full_name, "stars"] = info["stars"]
        df_out.loc[df_out["owner_repo"] == full_name, "forks"] = info["forks"]

        # Save after each repo
        df_out.to_csv(OUTPUT_FILE, index=False)

        print(
            f"[{i + 1}/{len(df)}] {full_name}: ⭐ {info['stars']} | 🍴 {info['forks']}"
        )

        time.sleep(1)  # polite delay

    print(f"\nDone! Saved enriched data to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
INPUT_FILE = "pymatgen_dependents_with_stars.csv"

df = pd.read_csv(INPUT_FILE)

# Ensure stars is numeric
df["stars"] = pd.to_numeric(df["stars"], errors="coerce").fillna(0).astype(int)

# Sort by stars (descending)
df = df.sort_values("stars", ascending=False)
df.to_csv(INPUT_FILE, index=False)

total_stars = df["stars"].sum()
print(f"Total stars across all dependents: {total_stars}")

print("\nTop 20 dependent repos by stars:")
print(df.head(20)[["owner_repo", "stars", "forks", "repo_url"]].to_string(index=False))

Total stars across all dependents: 20360

Top 20 dependent repos by stars:
                                     owner_repo  stars  forks                                                           repo_url
            google-deepmind/materials_discovery   1047    166             https://github.com/google-deepmind/materials_discovery
                           deepmodeling/Uni-Mol    944    151                            https://github.com/deepmodeling/Uni-Mol
                                   divelab/AIRS    688     79                                    https://github.com/divelab/AIRS
                  materialsintelligence/mat2vec    630    184                   https://github.com/materialsintelligence/mat2vec
                     materialsvirtuallab/megnet    539    157                      https://github.com/materialsvirtuallab/megnet
                            microsoft/mattersim    460     59                             https://github.com/microsoft/mattersim
                      